In [0]:
%run ./00_Config_Setup

In [0]:
# Notebook: 03_Gold_Jobs

# Leitura das Silvers
df_yellow = spark.table(f"{catalog}.{schema}.silver_yellow")
df_fhv = spark.table(f"{catalog}.{schema}.silver_fhv")

# União Global e Enriquecimento com Broadcast
df_zones = spark.read.format("csv").option("header", True).load(path_zones)

df_gold = df_yellow.unionByName(df_fhv, allowMissingColumns=True)

# Transformações de Data (Substituindo a necessidade do Join por derivação direta)
# Extraímos ano e mês diretamente da pickup_datetime, que já foi limpa na Silver.
df_final = (df_gold
    .withColumn("year", F.year("pickup_datetime"))
    .withColumn("month", F.month("pickup_datetime"))
    # Filtro de segurança: removemos outliers de datas futuras ou muito antigas
    .filter((F.col("year") >= 2009) & (F.col("year") <= 2019))
)
# Escrita Final Particionada
df_final.write.format("delta").mode("overwrite").partitionBy("year").saveAsTable(f"{catalog}.{schema}.gold_mobility_ny")


In [0]:

# Criando a agregação de Market Share
df_market_share = spark.sql(f"""
    WITH mensal_agrupado AS (
        SELECT 
            year, 
            month, 
            taxi_type, 
            COUNT(*) as total_viagens
        FROM {catalog}.{schema}.gold_mobility_ny
        WHERE year BETWEEN 2009 AND 2019
        GROUP BY 1, 2, 3
    ),
    total_mes AS (
        SELECT 
            year, 
            month, 
            SUM(total_viagens) as volume_total_mercado
        FROM mensal_agrupado
        GROUP BY 1, 2
    )
    SELECT 
        m.year, 
        m.month, 
        m.taxi_type,
        m.total_viagens,
        ROUND((m.total_viagens / t.volume_total_mercado) * 100, 2) as market_share_pct,
        CAST(CONCAT(m.year, '-', LPAD(m.month, 2, '0'), '-01') AS DATE) as reference_date
    FROM mensal_agrupado m
    JOIN total_mes t ON m.year = t.year AND m.month = t.month
""")

# salvando com uma tabela gold de consumo
df_market_share.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.gold_market_share_summary")
                                                                    
print("Tabela de Market Share criada com sucesso!")
